<br/>
<img src="images/utfsm.png" alt="" width="130px" align="left"/>
<img src="images/utfsm.png" alt="" width="130px" align="right"/>
<div align="center">
<h1>Deep Reinforcement Learning</h1>
<br/><br/>
Dr. Nicolás Gálvez Ramírez<br/>
Dr. Patricio Olivares Roncagliolo<br/><br/>
Ingeniería Civil Telemática<br/>
Departamento de Electrónica<br/>
Universidad Técnica Federico Santa María
</div>
<br>
Fuentes: 
<br>
"Hands-on Machine Learning with Scikit-Learn, Keras & TensorFlow"
<br>
"Standford - CS229 Notes"

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Comprender** el paradigma agente–entorno y la formulación de un MDP (estados, acciones, recompensas, transición, γ).
2. **Aplicar** la ecuación de Bellman y las funciones de valor y de acción-valor (Q).
3. **Implementar** Q-Learning tabular en un entorno de mundo de grilla.
4. **Motivar** y describir las Deep Q-Networks (DQN): replay buffer y target network.
5. **Entrenar** y evaluar un agente de aprendizaje por refuerzo.

> 💡 **Prerrequisitos:** [04_RedesNeuronales](../../04_RedesNeuronales/04_RedesNeuronales.ipynb).

## 1. Reinforcement Learning (Aprendizaje por Refuerzo)

- Reinforcement Learning es un conjunto de modelos y algoritmos que permiten resolver problemas de tomas de decisiones.
- Se enfocan en obtener el conjunto de acciones que permitan obtener la mejor salida posible.
- A diferencia de los modelos de aprendizaje supervisado, en estos casos no hay una solución inequívoca o una única respuesta correcta.
- Ejemplos
    - En una partida de ajedrez, no existe un único conjunto de movimientos que permita ganar una partida.
    - Si queremos que un robot cuadrúpedo camine, no tenemos conocimiento previo de cuáles son las acciones correctas que le permitan caminar.
    - [DeepMind](https://www.youtube.com/watch?v=V1eYniJ0Rnk)

Elementos principales de modelos RL

- **Agente**: Es el elemento encargado de la toma de decisiones. Además es el responsable de aprender qué acciones tomar para lograr el mejor resultado.
- **Entorno**: Es el mundo en el que opera el agente. Este puede ser real, virtual o cualquier otro elemento con el que el agente interactúe.
- **Acciones**: Cada agente puede realizar acciones las cuales les permiten interactuar con su entorno.
- **Estado**: Corresponde a una representación del entorno en un cierto momento. El agente utiliza el estado como contexto para la toma de decisiones y sus correspondientes acciones
- **Recompensa**: Las recompensas son recibidas por el agente desde el entorno en base a cada acción tomada. Las recompensas pueden ser positivas o negativas en base a un objetivo
- **Política**: La política es una estrategia que el agente sigue para decidir qué acciones tomar en función de su estado actual. El objetivo del agente es aprender una política óptima que le permita obtener la máxima recompensa posible a lo largo del tiempo.

## 2. Formalización: Procesos de Decisión de Markov (MDP)

Definimos un Proceso de Decisión de Markov (Markov Decision Process) como una tupla $(S, A, \{P_{sa}\}, \gamma, R)$ donde

- $S$ es el conjunto estados en que se puede encontrar el entorno
- $A$ es el conjunto de acciones posible que puede realizar el agente
- $P_{sa}$ son las probabilidades de transición entre estados. Estas se definen para cada estado $s \in S$ y acción $a \in A$. En otras palabras, es la distribución de probabilidades sobre cada estado $s$ al tomar una acción $a$.
- $\gamma \in [0,1)$ es el factor de descuento
- $R:S \times A \rightarrow \mathbb{R}$ es la función de recompensa. En ocasiones se escribe como una función solo dependiente del estado, en cuyo caso se anota como $R:S \rightarrow \mathbb{R}$ 

Las dinámicas de un proceso de Markov quedan definidas de la siguiente manera:

- Se define un estado inicial $s_0 \in S$, a partir del cual se toma una acción $a_0 \in A$.
- Como resultado de la decisión $a_0$, se transiciona aleatoriamente a un estado $s_1$ donde $s_1 \sim P_{s_0 a_0}$.
- A partir del estado $s_1$, se toma una nueva decisión $a_1$.
- La decisión $a_1$ genera una transición aleatoria a un nuevo estado $s_2 \sim P_{s_1 a_1}$

<img src="images/MDP_diagram.jpg" alt="" width="800px" align="center"/>

Para este caso la recompensa total puede ser definida como

$$ R(s_0, a_0)  + \gamma R(s_1, a_1) + \gamma^2 R(s_2, a_2)+ \cdots$$

o, reescribiendo como función dependiente de los estados

$$ R(s_0)  + \gamma R(s_1) + \gamma^2 R(s_2)+ \cdots$$

El objetivo del aprendizaje por refuerzo es **determinar el conjunto de acciones que maximizan la esperanza de la recompensa**

$$ E [ R(s_0)  + \gamma R(s_1) + \gamma^2 R(s_2)+ \cdots ]$$

- Considere una **función de política** $\pi: S \rightarrow A$, la cual mapea cada estado a una acción. Es posible definir la **función de valor** $V^{\pi}(s)$, que corresponde a la recompensa esperada acumulada al seguir la política $\pi$ desde un cierto estado inicial $s$:

$$ V^{\pi}(s) =  E [ R(s_0)  + \gamma R(s_1) + \gamma^2 R(s_2)+ \cdots \mid s_0 = s, \pi]$$

- Esta puede reescribirse en su forma recursiva, conocida como **ecuación de Bellman**:

$$ V^{\pi}(s) = R(s) + \gamma \sum_{s' \in S} P_{s\,\pi(s)}(s') \, V^{\pi}(s') $$

## 3. Mundo Grilla (Grid World)

Consideremos el siguiente caso.

- Movimientos posibles: Arriba ($\uparrow$), abajo ($\downarrow$), izquierda ($\leftarrow$) y derecha ($\rightarrow$)
- Casillas +1 y -1 corresponden a las casillas de salida con recompensas del mismo valor
- Objetivo: terminar el juego con la mayor puntuación posible

<img src="images/MundoGrilla.drawio.png" alt="" width="400px" align="center"/>

Se define la siguiente política $\pi$ para cada estado de la grilla

<img src="images/MundoGrilla_politica.drawio.png" alt="" width="400px" align="center"/>

### 3.1 Caso determinista (sin descuento)

- Mejores movimientos desde inicio: $\uparrow, \uparrow, \rightarrow, \rightarrow, \rightarrow$

<img src="images/MundoGrilla_determinista.drawio.png" alt="" width="400px" align="center"/>

### 3.2 Caso determinista (con descuento)

<img src="images/MundoGrilla_determinsta_descuentos.drawio.png" alt="" width="400px" align="center"/>

### 3.3 Caso probabilístico (estocástico)

- Los casos anteriores representan situaciones deterministas donde cada acción a realizar por el agente tiene un 100% de probabilidad de éxito (sin errores).
- La ecuación de Bellman permite también considerar casos en donde el agente puede cometer errores.

<img src="images/MDP_estocastico_vs_determinista.png" alt="" width="400px" align="center"/>

Considere ahora que los movimientos del agente no son confiables, es decir **tienen cierta probabilidad**.
Considere las siguientes probabilidades:
- $80\%$ de ir a la dirección deseada
- $10\%$ de ir a la izquierda de la dirección deseada
- $10\%$ de ir a la derecha de la dirección deseada

Ejemplo:
- $P(\uparrow|\uparrow)=0.8,  P(\leftarrow|\uparrow)=0.1, P(\rightarrow|\uparrow)=0.1$

**¿Cómo podemos calcular las recompensas de cada casilla?** R: Algoritmo Value Iteration

<img src="images/MundoGrilla_probabilistica.drawio.png" alt="" width="400px" align="center"/>

## 4. Algoritmo Value Iteration

- Para cada estado $s$, inicializar $V(s):=0$
- Repetir hasta la convergencia lo siguiente:
    - Por cada estado, actualizar $V(s):=R(s) + \max_{a \in A} \gamma \sum_{s'} P_{sa}(s')V(s')$


Esto no solo permite calcular los valores de recompensa esperada por cada estado, sino que también permite obtener la mejor política para el conjunto de estados (*policy iteration*):

- Inicializar $\pi$ aleatoriamente
- Repetir hasta la convergencia lo siguiente:
    - Por cada estado $s$: $\pi(s):= \arg\max_{a \in A} \sum_{s'} P_{sa}(s')V(s')$

## 5. Q-Learning

- Hasta el momento, hemos supuesto que las recompensas y las probabilidades de acción son conocidas. ¿Cómo cambia nuestro planteamiento si no conocemos estos datos?
- En ese caso, debemos descubrir sus valores en base a la **experiencia** (muestras).
- Para ello calcularemos, para cada combinación de estado/acción, cuáles son las recompensas esperadas en base a las muestras (valores $Q$). Estos se calculan con la siguiente fórmula:

    $$Q(s,a) := Q(s,a) + \alpha [ R(s,a) + \gamma \max_{a'}\{ Q(s',a') \} - Q(s,a) ] $$

Donde encontramos la siguiente relación entre los valores $Q$ y los elementos previamente calculados:

$$ V(s) = \max_{a}\left(R(s,a) + \gamma \sum_{s'} P_{sa}(s') V(s')\right) = \max_{a} Q(s,a)$$
$$ \pi(s) = \arg\max_{a} Q(s,a)$$

- $Q$ viene de *quality*, pues mide la calidad de las acciones que tomamos a partir de un cierto estado.

## 6. Deep Q-Learning (DQN)

- Si bien, el modelo Q Learning funciona correctamente, no escala bien en relación a la cantidad de estados y acciones.
- Por ello, se plantean los modelos de redes neuronales como alternativa para la obtención de los valores Q (Deep Q Learning).
- Los valores obtenidos por el algoritmo Q Learning, son utilizados como base para una red neuronal. Esta red neuronal recibe como entrada los valores del estado del entorno, y retorna los valores Q para todas las acciones disponibles.

<img src="images/q-learning.png" alt="" width="400px" align="center"/>
<img src="images/deep-q-learning.png" alt="" width="400px" align="center"/>

Source: https://www.geeksforgeeks.org/deep-q-learning/

### 6.1 Funcionamiento de Deep Q-Learning (DQN)

* En **Deep Q-Learning**, una **red neuronal** se utiliza para aproximar los valores $Q(s,a)$ en lugar de almacenarlos en una tabla.
* Para cada par de **estado–acción** $(s,a)$, la red predice un **Q-Value estimado**, que representa qué tan buena es esa acción en ese estado.
* Usando la **ecuación de Bellman**, podemos definir un valor objetivo (*target*) que queremos que la red aprenda:

$$Q_{\text{target}}(s,a) = R(s,a) + \gamma \max_{a'} Q_{\theta}(s', a')$$

* Este $Q_{\text{target}}$ combina la **recompensa inmediata** $R(s,a)$ con el **mejor valor futuro estimado** desde el nuevo estado $s'$.
* El entrenamiento de la red consiste en **ajustar sus pesos** $\theta$ para minimizar la diferencia entre su predicción actual $Q_{\theta}(s,a)$ y el valor objetivo $Q_{\text{target}}(s,a)$.

Para estabilizar el entrenamiento, DQN incorpora dos mecanismos clave:

* **Experience Replay (replay buffer):** las transiciones $(s, a, r, s')$ se almacenan en una memoria y se muestrean minilotes aleatorios para entrenar, rompiendo la correlación temporal entre muestras consecutivas.
* **Red objetivo (target network):** una copia de la red $Q_{\theta}$ cuyos pesos se actualizan de forma periódica y se usa para calcular $Q_{\text{target}}$, evitando que el objetivo cambie en cada paso y estabilizando la convergencia.

## 7. OpenAI Gymnasium

- Biblioteca Python de OpenAI (originalmente llamada Gym)
- Permite con una interfaz sencilla interactuar con diferentes entornos para la prueba de modelos de Aprendizaje por Refuerzo utilizando una API común para todos ellos.
- Se puede instalar en un entorno virtual a través de pip con `pip install gymnasium`
- Gymnasium posee múltiples entornos los que se pueden instalar manualmente. Si desea instarlos todos, puede hacerlo a través del comando pip a continuación: `pip install gymnasium[all]`
- Documentación disponible [aquí](https://gymnasium.farama.org/)

### 7.1 Ejemplo de uso de Gymnasium

El siguiente ejemplo muestra el ciclo básico de interacción agente–entorno: crear el entorno, reiniciarlo, muestrear acciones y avanzar con `env.step()`. En este caso las acciones son aleatorias (aún no hay aprendizaje).

In [ ]:
# Se importa la biblioteca
import gymnasium as gym
# Se crea uno de los entornos a través de make
# Se puede obtener un renderizado (visualización) del juego a través de
# .make("EnvName", render_mode="human")
env = gym.make("CartPole-v1", render_mode="human")

# Una vez creado el entorno, obtenemos el estado inicial (primera observación)
# usando env.reset()
observation, info = env.reset()

# El entorno mantiene almacenado los valores posibles tanto de acciones como de observaciones (estados)
# Las acciones se mantienen en env.action_space y las observaciones en env.observation_space
for _ in range(300):
    # Probamos nuestro entorno con acciones aleatorias
    # A partir del action_space obtenemos una muestra de una acción
    action = env.action_space.sample()  # agent policy that uses the observation and info
    
    # Utilizamos esa acción para el siguiente step de evaluación. Como resultado
    # obtenemos una nueva observación del estado del entorno y una recompensa.
    # La función env.step(action) devuelve:
    # - observation: el nuevo estado del entorno tras ejecutar la acción.
    # - reward: la recompensa obtenida por dicha acción.
    # - terminated: True si el episodio terminó porque se alcanzó un estado final.
    # - truncated: True si el episodio se detuvo por límite de tiempo u otra condición.
    # - info: diccionario con información adicional del entorno.
    observation, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        observation, info = env.reset()

# Finalmente cerramos el entorno al terminar la ejecución        
env.close()

In [ ]:
# DQN sencillo para CartPole-v1 (Gymnasium)

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # (opcional) usar GPU 0

import random
from collections import deque
import numpy as np
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Ajustes de GPU para utilizar solo la memoria vram necesaria (opcional)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print("Aviso TF:", e)

# Hiperparámetros
ENV_NAME = "CartPole-v1"
GAMMA = 0.99
LR = 1e-3
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.01, 0.995
BATCH_SIZE = 64
MEM_CAPACITY = 50000
TARGET_UPDATE_EVERY = 200  # pasos
EPISODES = 300
MAX_STEPS = 500
CHECKPOINT_PATH = "dqn_cartpole.weights.h5"
SAVE_EVERY = 50  # guardar cada 50 episodios

# Red Q y Red Target
def build_q_network(input_dim=4, n_actions=2, lr=LR):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,)),
        Dense(64, activation='relu'),
        Dense(n_actions, activation='linear')
    ])
    model.compile(optimizer=Adam(lr), loss=tf.keras.losses.Huber())
    return model

# Utilidades DQN
class ReplayBuffer: # memoria
    def __init__(self, capacity):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, ns, done):
        self.buf.append((s, a, r, ns, done))
    def sample(self, batch_size):
        minibatch = random.sample(self.buf, batch_size)
        s, a, r, ns, d = map(np.array, zip(*minibatch))
        # Asegurar shapes (N,4) y (N,)
        return (np.vstack(s), a, r.astype(np.float32),
                np.vstack(ns), d.astype(np.float32))
    def __len__(self):
        return len(self.buf)

# Selección de acciones 
def epsilon_greedy_action(model, state, epsilon, n_actions):
    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)
    q = model.predict(state[None, :], verbose=0)
    return int(np.argmax(q[0]))

# Actualización de red
def hard_update(target_model, online_model):
    target_model.set_weights(online_model.get_weights())

# Entrenamiento
env = gym.make(ENV_NAME)
n_actions = env.action_space.n
state_dim = env.observation_space.shape[0]

q_net = build_q_network(state_dim, n_actions)
target_net = build_q_network(state_dim, n_actions)
hard_update(target_net, q_net)

buffer = ReplayBuffer(MEM_CAPACITY)

epsilon = EPS_START
global_step = 0

for ep in range(1, EPISODES + 1):
    state, _ = env.reset()
    total_reward = 0.0

    for t in range(MAX_STEPS):
        global_step += 1

        # 1) Política ε-greedy para selección de acción
        action = epsilon_greedy_action(q_net, state, epsilon, n_actions)

        # 2) Paso en el entorno aplicando acción seleccionado
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward

        # 3) Guardar transición en la memoria
        buffer.push(state, action, reward, next_state, float(done))

        # 4) Verifica si hay muestras suficientes en el buffer para entrenar el modelo
        if len(buffer) >= BATCH_SIZE:
            s, a, r, ns, d = buffer.sample(BATCH_SIZE)

            # Q target: r + gamma * (1-done) * max_a' Q_target(ns, a')
            q_next = target_net.predict(ns, verbose=0)              # (N, n_actions)
            max_q_next = np.max(q_next, axis=1)                     # (N,)
            target = r + GAMMA * (1.0 - d) * max_q_next             # (N,)

            # Q online actual
            q_curr = q_net.predict(s, verbose=0)                    # (N, n_actions)
            q_curr[range(BATCH_SIZE), a] = target                   # actualizar solo la acción tomada

            # Fit un paso
            q_net.fit(s, q_curr, epochs=1, verbose=0, batch_size=BATCH_SIZE)

        # 5) Actualización periódica de la red target
        if global_step % TARGET_UPDATE_EVERY == 0:
            hard_update(target_net, q_net)

        state = next_state
        if done:
            break

    # Decaimiento de epsilon por episodio
    epsilon = max(EPS_END, epsilon * EPS_DECAY)

    # Guardar checkpoint periódicamente
    if ep % SAVE_EVERY == 0:
        q_net.save_weights(CHECKPOINT_PATH)
        print(f"Checkpoint guardado en {CHECKPOINT_PATH}")

    print(f"Episodio {ep:3d} | Recompensa: {total_reward:5.1f} | ε={epsilon:.3f}")

env.close()


In [ ]:
# Evaluación del DQN entrenado en CartPole-v1
# Carga pesos guardados y prueba el agente

import numpy as np
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Configuración
ENV_NAME = "CartPole-v1"
CHECKPOINT_PATH = "dqn_cartpole.weights.h5"
MAX_STEPS = 500

# Reconstruir la red
# (debe coincidir exactamente con la del entrenamiento)
def build_q_network(input_dim=4, n_actions=2, lr=1e-3):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,)),
        Dense(64, activation='relu'),
        Dense(n_actions, activation='linear')
    ])
    model.compile(optimizer=Adam(lr), loss=tf.keras.losses.Huber())
    return model

# Cargar pesos
env = gym.make(ENV_NAME, render_mode="human")  # usa "None" si no quieres render
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

model = build_q_network(state_dim, n_actions)
model.load_weights(CHECKPOINT_PATH)
print(f"Pesos cargados desde {CHECKPOINT_PATH}")

# Evaluar episodios
def evaluate(episodes=10):
    rewards = []
    for ep in range(episodes):
        s, _ = env.reset()
        ep_reward = 0.0
        for _ in range(MAX_STEPS):
            q = model.predict(s[None, :], verbose=0)
            a = int(np.argmax(q[0]))  # política greedy (mejor acción)
            s, r, term, trunc, _ = env.step(a)
            ep_reward += r
            if term or trunc:
                break
        rewards.append(ep_reward)
        print(f"Episodio {ep+1}: Recompensa = {ep_reward:.1f}")
    print(f"\nRecompensa promedio en {episodes} episodios: {np.mean(rewards):.1f}")
    return rewards

evaluate(episodes=10)
env.close()


In [ ]:
# Utilidad de presentación: centra las imágenes de matplotlib en las diapositivas.
# Ejecutar antes de lanzar los ejemplos.
from IPython.core.display import HTML
HTML("""
<style>
.output_png {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")
#codigo extra, para que imagenes de matplotlib
#estén centradas en las diapositivas, ejecutar antes de lanzar los ejemplos.

## 📌 Resumen

| Concepto | Descripción breve |
|----------|-------------------|
| **Agente / Entorno** | El agente toma acciones; el entorno responde con un nuevo estado y una recompensa. |
| **Exploración vs. explotación** | Compromiso entre probar acciones nuevas y aprovechar las que ya dan buena recompensa. |
| **MDP $(S, A, \{P_{sa}\}, \gamma, R)$** | Marco formal para la toma de decisiones secuenciales. |
| **Factor de descuento $\gamma$** | Pondera la importancia de las recompensas futuras frente a las inmediatas. |
| **Ecuación de Bellman** | Relación recursiva que expresa el valor de un estado en función de sus sucesores. |
| **Función de valor $V^{\pi}(s)$** | Recompensa esperada acumulada al seguir la política $\pi$ desde $s$. |
| **Función de acción-valor $Q(s,a)$** | Calidad esperada de tomar la acción $a$ en el estado $s$. |
| **Value Iteration** | Calcula $V$ y la política óptima cuando el modelo ($P$, $R$) es conocido. |
| **Q-Learning** | Aprende $Q$ a partir de la experiencia, sin conocer $P$ ni $R$. |
| **Deep Q-Learning (DQN)** | Aproxima $Q(s,a)$ con una red neuronal; usa *replay buffer* y *target network*. |
| **Gymnasium** | Biblioteca con entornos estandarizados para entrenar y evaluar agentes de RL. |

## 🔗 Próximos pasos
- Extensiones: Double DQN, Dueling DQN y métodos de gradiente de política (REINFORCE).
- Explorar otros entornos de Gymnasium (LunarLander, MountainCar).

## 📚 Referencias

- **Sutton, R. S. & Barto, A. G. (2018).** *Reinforcement Learning: An Introduction* (2ª ed.). MIT Press. http://incompleteideas.net/book/the-book.html
- **Mnih, V. et al. (2015).** *Human-level control through deep reinforcement learning*. Nature, 518, 529–533. https://doi.org/10.1038/nature14236
- **Géron, A. (2019).** *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (2ª ed.). O'Reilly Media. Cap. 18: Reinforcement Learning.
- **Ng, A. — Stanford CS229.** *Reinforcement Learning and Control* (notas de curso). https://cs229.stanford.edu/
- **Gymnasium (Farama Foundation).** Documentación oficial: https://gymnasium.farama.org/